- #### Libs & Functions 

In [ ]:
# import packages
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, RocCurveDisplay
import numpy as np
from sklearn.metrics import precision_recall_curve, f1_score, precision_score, recall_score, confusion_matrix, classification_report

- ####  Global Parameters

- ####  Read Data

In [ ]:
path = "C:/Users/clamo/Documents/Doutorado/HC/hc_models/features/"

In [ ]:
internacao = pd.read_parquet(path + "sample_features_internacao.parquet").reset_index(drop=True)
targets = pd.read_parquet(path + "sample_target_internacao.parquet").reset_index(drop=True)
evolucoes = pd.read_parquet(path + "sample_evolucao.parquet").reset_index(drop=True)
paciente = pd.read_parquet(path + "pacientes.parquet").reset_index(drop=True)

In [ ]:
pacientes.head()

In [ ]:
targets.head()

In [ ]:
data = targets.join(internacao, on=["prontuario", "date_ref"], how='left')
data= data.join(evolucoes, on=["prontuario", "date_ref"],  how='left')
data.shape

In [0]:
print(targets.select("target_180_empretec").groupBy("target_180_empretec").count().show())

In [0]:
print(data.select("target_180_empretec").groupBy("target_180_empretec").count().show())

In [0]:
# Verificação do balanço dos dados 
balanco = (data.agg (
           (F.sum(F.col("target_90_empretec"))/F.count("*")).alias("balanco_target_90"),
           (F.sum(F.col("target_180_empretec"))/F.count("*")).alias("balanco_target_180") 
           )).show()

In [0]:
# Verificando todos os valores distintos e suas contagens para uma das colunas
data.groupBy("target_90_empretec").count().show()
data.groupBy("target_180_empretec").count().show()

In [0]:
# total rows (avoid recomputing repeatedly)
total_rows = data.count()

# count nulls and compute percentage
null_stats = data.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in data.columns
])

# add percentages
null_stats_percent = null_stats.select([
    F.col(c).alias(c + "_nulls") for c in data.columns
] + [
    (F.col(c) / total_rows * 100).alias(c + "_pct") for c in data.columns
])

display(null_stats_percent)

In [0]:
#data = data.toPandas()

<h3> Split Temporal <h3>

In [0]:
# Criando nova coluna de mês/ano
df = data
df = df.withColumn("yyyymm", F.date_format("date_ref", "yyyy-MM"))
display(df)

In [0]:
data.printSchema()

In [0]:
df = data.select(
    "date_ref",
    "target_180_empretec",
    "flag_cliente_ativo_historico_realizacoes",
    "qtd_atendimentos_total",
    "qtd_servicos_distintos",
    "qtd_instrumento_orientacao",
    "qtd_instrumento_oficina",
    "qtd_instrumento_palestra",
    "qtd_instrumento_seminario",
    "qtd_instrumento_curso",
    "qtd_instrumento_ferramenta",
    "qtd_instrumento_consultoria",
    "qtd_instrumento_missao_caravana",
    "perc_instrumento_orientacao",
    "perc_instrumento_oficina",
    "perc_instrumento_palestra",
    "perc_instrumento_seminario",
    "perc_instrumento_curso",
    "perc_instrumento_ferramenta",
    "perc_instrumento_consultoria",
    "perc_instrumento_missao_caravana",
    "cliente_tempo_media_resposta",
    "cliente_areas_interesse",
    "date_primeiro_atendimento",
    "val_recencia",
    "val_frequencia_media_mensal",
    "qtd_eventos_participados",
    "flag_cliente_ativo_historico_eventos",
    "val_total_carga_horaria_eventos",
    "val_media_carga_horaria_eventos",
    "flag_gendercodename_mas",
    "flag_gendercodename_fem",
    "val_idade"
)


# Realizando corte nas datas a fim de evitar leaks e separando em dados de treino, validação e teste
w = Window.orderBy(F.col("date_ref").asc())
df_split = df.withColumn("bucket_3", F.ntile(3).over(w))

train = df_split.filter(F.col("bucket_3")==1).drop("bucket_3")
valid = df_split.filter(F.col("bucket_3")==2).drop("bucket_3")
test  = df_split.filter(F.col("bucket_3")==3).drop("bucket_3")

In [0]:
train =train.toPandas()

In [0]:

# 1. Remove colunas que não são numéricas ou booleanas
train = train.select_dtypes(include=["number", "bool"]).copy()

# 2. Substitui infinitos por NaN
train = train.replace([np.inf, -np.inf], np.nan)

# --- data ---
# df = pd.read_csv("data.csv")
target = "target_180_empretec"
X = train.drop(columns=[target] + ([ 'date_ref'] if 'date_ref' in train.columns else [])).copy()
y = train[target].astype(int)

X = X.select_dtypes(include=["number", "bool"]).copy()

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [0]:
X.columns

In [0]:
# --- model ---
clf = LGBMClassifier(random_state=42)
clf.fit(Xtr, ytr)

# --- AUROC ---
proba = clf.predict_proba(Xte)[:, 1]
auc = roc_auc_score(yte, proba)
print(f"AUROC: {auc:.4f}")

# --- ROC curve (optional) ---
RocCurveDisplay.from_predictions(yte, proba)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Gráfico de densidade dos scores preditos ---
plt.figure(figsize=(8, 5))
sns.kdeplot(proba[yte == 0], label="Classe 0", fill=True)
sns.kdeplot(proba[yte == 1], label="Classe 1", fill=True)
plt.title("Distribuição das probabilidades preditas (densidade)")
plt.xlabel("Probabilidade prevista da classe positiva")
plt.ylabel("Densidade")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()


In [0]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    PrecisionRecallDisplay
)

# --- Predictions ---
preds = clf.predict(Xte)
proba = clf.predict_proba(Xte)[:, 1]

# --- Metrics ---
auc = roc_auc_score(yte, proba)
acc = accuracy_score(yte, preds)
prec = precision_score(yte, preds, zero_division=0)
rec = recall_score(yte, preds)
f1 = f1_score(yte, preds)

print(f"AUROC:   {auc:.4f}")
print(f"Accuracy:{acc:.4f}")
print(f"Precision:{prec:.4f}")
print(f"Recall:  {rec:.4f}")
print(f"F1-score:{f1:.4f}")

# --- Confusion Matrix ---
print("\nConfusion Matrix:")
print(confusion_matrix(yte, preds))

# --- Full Classification Report ---
print("\nClassification Report:")
print(classification_report(yte, preds, digits=4))

# --- Precision-Recall Curve (useful for imbalance) ---
PrecisionRecallDisplay.from_predictions(yte, proba)


In [0]:

# proba = clf.predict_proba(Xte)[:, 1]
# yte = yte

# --- 1) Threshold que maximiza F1 ---
prec, rec, thr = precision_recall_curve(yte, proba)
# obs: len(thr) = len(prec) - 1 = len(rec) - 1
f1_vals = (2 * prec[1:] * rec[1:]) / (prec[1:] + rec[1:] + 1e-12)
i_best = np.argmax(f1_vals)
best_thr = thr[i_best]

preds_best = (proba >= best_thr).astype(int)
print(f"[F1 Máximo] threshold = {best_thr:.4f}")
print(f"Precision={precision_score(yte, preds_best, zero_division=0):.4f} | "
      f"Recall={recall_score(yte, preds_best):.4f} | "
      f"F1={f1_score(yte, preds_best):.4f}")
print("Confusion Matrix:\n", confusion_matrix(yte, preds_best))
print("\nClassification Report:\n", classification_report(yte, preds_best, digits=4))

# --- 2) Threshold com recall mínimo (ex.: >= 0.70) e melhor F1 entre eles ---
recall_min = 0.70
candidates = np.where(rec[1:] >= recall_min)[0]
if len(candidates) > 0:
    j = candidates[np.argmax(f1_vals[candidates])]
    thr_rec = thr[j]
    preds_rec = (proba >= thr_rec).astype(int)
    print(f"\n[Recall ≥ {recall_min:.2f}] threshold = {thr_rec:.4f}")
    print(f"Precision={precision_score(yte, preds_rec, zero_division=0):.4f} | "
          f"Recall={recall_score(yte, preds_rec):.4f} | "
          f"F1={f1_score(yte, preds_rec):.4f}")
    print("Confusion Matrix:\n", confusion_matrix(yte, preds_rec))
else:
    print(f"\nNenhum threshold alcança recall ≥ {recall_min:.2f}. "
          "Tente reduzir o limite ou ajustar class_weight/params do modelo.")
